### Step 1:- Imports 
- FAISS -> langchain_community.vectorstores
- pandas
- BM25Retriever -> langchain_community.retrievers
- EnsembleRetriever -> langchain_community.retrievers
- HuggingFaceEmbeddings -> langchain_community.embeddings
- Document -> langchain_core.documents


In [1]:
import pandas as pd
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

C:\Users\Vikrant Singh Thakur\AppData\Local\Temp\ipykernel_18944\3066272891.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
c:\Users\Vikrant Singh Thakur\OneDrive\Documents\nlpFinal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2:- 
- Create a variable called model and create an instance of BAAI/bge-base-en-v1.5
using proper parameters.
- model_name, model_kwargs, encode_kwargs

In [2]:
model=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5", 
model_kwargs={"device":"cpu"}, 
encode_kwargs={"normalize_embeddings":True})

C:\Users\Vikrant Singh Thakur\AppData\Local\Temp\ipykernel_18944\4254936896.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model=HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5",
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4440.24it/s]


## Step 3:- 
- load in the local faiss stuff in the location "../../data/EMBEDDINGS&FAISS"
- use load_local()
- Parameter to define in load_local():-
    - folder_path=""
    - embeddings=
    - allow_dangerous_deserialization=

In [3]:
faiss_vectors=FAISS.load_local(
    folder_path="../../data/EMBEDDINGS&FAISS",
    embeddings=model,
    allow_dangerous_deserialization=True
)

## Step 4:- 
- Create a retriever for FAISS and store it in a variable.
- use local_load()
- define folder_path= "", embeddings= , allow_dangerous_deserialization=True in local_load.

In [4]:
# faiss_retriever=faiss_vectors.

## Step 5:- 
- Create a variable for faiss retriever.
- use variableNameOfVectorLibInStep3.as_retriever()
- In that, define search_kwargs={k=} for fetching the top results.

In [5]:
faiss_retriever=faiss_vectors.as_retriever(search_kwargs={"k":3})

## Step 6:- 
- load in the csv containing your chunks.

In [6]:
df=pd.read_csv("../../data/chunked/langChainchunks.csv")

#####################################################################################################################
                                                    BM25
#####################################################################################################################

## Step 7:- 
- Create an empty list.

In [7]:
em=[]

## Step 8:- 
- Create a for loop which iterates over the csv rows to get index and row content.
- Use Document. and get the chunks and metadata.
- append the chunks and metadata to the empty list.

In [8]:
for i, j in df.iterrows():
    doc=Document(page_content=j["chunk_text"],
        metadata={
            "source": j["source"],
            "url": j["url"],
            "title": j["title"]
        }
    )
    em.append(doc)
    

## Step 9:- 
- Make a variable and in that variable, call BM25Retriever and pass in the document as the list which we created in step 7, also do a preprocess_func=str.lower

In [9]:
BM25_actual=BM25Retriever.from_documents(em, preprocess_func=str.lower)

## Step 10:- 
- On the same variable in step 9, define a top return result. (k)

In [10]:
BM25_actual.k=3

## Step 11:- 
- Create a variable for Ensemble Retriever.
- In it use Langchain's EnsembleRetriever and in it define retrievers as both BM25 and faiss_retriever in a list, weights also in a list. (weight total can be only upto 1.)

In [12]:
ensemble_retriever=EnsembleRetriever(retrievers=[faiss_retriever, BM25_actual], weights=[0.6,0.4])

## Step 12:- 
- Define a query

In [13]:
query="GAMING using NVIDIA and geforce now."

## Step 13:- 
- Create a variable and in it do a invoke for the query using variable we created in Step 11.

In [14]:
result=ensemble_retriever.invoke(query)

## Step 14:- 
- Create a for loop which returns index and the content using enumerate on the variable we created in step 13.
- print everything.

In [17]:
for i, doc in enumerate(result):
    print("\n--- Result", i+1, "---")
    print("Source:", doc.metadata["source"])
    print("Title:", doc.metadata["title"])
    print("Text preview:")
    print(doc.page_content[:400])


--- Result 1 ---
Source: NVIDIA Blog
Title: Save Big and Play Bigger: GeForce NOW Summer Sale Brings Major Membership Savings
Text preview:
The GeForce NOW summer sale kicked off today with limited-time savings of up to $70 off a 12-month membership, making now the perfect time to upgrade to get the best of the cloud and see just how far Ultimate gaming can go. PC gamers are driven by one thing: the love of the game. But getting there can be complicated — setups take space, hardware takes planning and downloads take time. GeForce NOW 

--- Result 2 ---
Source: NVIDIA Blog
Title: Save Big and Play Bigger: GeForce NOW Summer Sale Brings Major Membership Savings
Text preview:
delivers smooth, high-quality cloud gaming across devices, with streaming up to 1080p at 60 frames per second (fps) and access to RTX-powered servers for supported games. The Ultimate membership steps things up with RTX 4080‑ or 5080‑class performance in the cloud, supporting up to 4K and beyond on ultrawide display